# Outcome-only concept model with a token-level late-interaction matcher

This version does **not** use note-level concept annotations for training. The matcher is fixed and ontology-informed: it compares clinical-note tokens with token embeddings of concise ICD-10 descriptions and aliases. Concept annotations are loaded only for held-out grounding evaluation.

The concept vocabulary is read from `df_icd10_with_synonyms.csv`, which must contain `code`, `name`, and `synonyms`. Each `synonyms` cell may be a Python/JSON-style serialized list such as `['rib fracture', 'rib fx']`.


In [ ]:
from __future__ import annotations

import json
import math
import os
import random
import re
from collections import Counter, defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import average_precision_score, roc_auc_score
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, AutoTokenizer

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(DEVICE, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
from from_n3c import *
import ast

CONCEPT_CSV = "../AdaptivePooling_MLHC/df_icd10_with_synonyms.csv"
TRAIN_JSON = "../AdaptivePooling_MLHC/ds_train_chest_trauma_ner.json"
DEV_JSON = "../AdaptivePooling_MLHC/ds_dev_chest_trauma_ner.json"
VAL_JSON = "../AdaptivePooling_MLHC/ds_test_chest_trauma_ner.json"
ALIAS_TABLE_OUT = "../AdaptivePooling_MLHC/icd10_3digit_aliases_generated.csv"

# Optional task-relevant additions; CSV synonyms remain the primary alias source.
CURATED_ALIASES = {
    "S22": ["rib fracture", "rib fractures", "broken ribs", "rib fx", "sternal fracture", "thoracic spine fracture"],
    "J93": ["pneumothorax", "collapsed lung", "PTX"],
    "J94": ["hemothorax", "haemothorax"],
    "S27": ["lung injury", "pulmonary injury", "intrathoracic injury"],
    "S42": ["clavicle fracture", "scapula fracture", "shoulder fracture"],
    "I95": ["hypotension", "low blood pressure"],
    "I46": ["cardiac arrest", "cardiopulmonary arrest"],
    "J96": ["respiratory failure", "acute respiratory failure", "resp failure"],
    "K72": ["hepatic failure", "liver failure"],
    "I50": ["heart failure", "congestive heart failure", "CHF"],
    "A41": ["sepsis", "septicemia"],
    "N17": ["acute kidney injury", "acute kidney failure", "AKI"],
    "R57": ["shock", "circulatory shock"],
}


def normalize_code(code: Any) -> str:
    return re.sub(r"[^A-Z0-9]", "", str(code).upper())[:3]


def normalize_alias(text: Any) -> str:
    return re.sub(r"\s+", " ", str(text)).strip(" ,;.")


def parse_synonyms(value: Any) -> List[str]:
    """Parse a serialized list while preserving commas inside individual terms."""
    if value is None:
        return []
    if not isinstance(value, (list, tuple, set, dict)):
        try:
            if pd.isna(value):
                return []
        except (TypeError, ValueError):
            pass

    parsed = value
    # Handles both "['a', 'b']" and a doubly quoted string containing that list.
    for _ in range(2):
        if not isinstance(parsed, str):
            break
        text = parsed.strip()
        if not text:
            return []
        new_value = None
        for candidate in (text, text.strip('"'), text.strip("'")):
            try:
                new_value = ast.literal_eval(candidate)
                break
            except (ValueError, SyntaxError):
                try:
                    new_value = json.loads(candidate)
                    break
                except (ValueError, TypeError, json.JSONDecodeError):
                    continue
        if new_value is None:
            parsed = re.split(r"[|;]", text)  # conservative fallback; do not split on commas
            break
        if new_value == parsed:
            break
        parsed = new_value

    if isinstance(parsed, dict):
        parsed = list(parsed.values())
    elif not isinstance(parsed, (list, tuple, set)):
        parsed = [parsed]

    synonyms = []
    for item in parsed:
        if item is None:
            continue
        item = normalize_alias(item)
        if item and item.casefold() not in {"nan", "none", "null"}:
            synonyms.append(item)
    return synonyms


def make_aliases(code: str, name: str, synonyms: Any) -> List[str]:
    candidates = [
        name,
        name.replace("(s)", "s"),
        re.sub(r"\[([^]]+)\]", r"\1", name),
        *parse_synonyms(synonyms),
        *CURATED_ALIASES.get(code, []),
    ]

    aliases, seen = [], set()
    for value in candidates:
        value = normalize_alias(value)
        key = value.casefold()
        if value and key not in seen:
            seen.add(key)
            aliases.append(value)
    return aliases


def concept_group(code: str) -> str:
    """Derive a stable group because the synonym CSV has no idx_section column."""
    try:
        return str(icd10_text(code))
    except Exception:
        try:
            return str(infer_chapter_from_code(code))
        except Exception:
            return code[:1]


df_concepts = pd.read_csv(CONCEPT_CSV)
required_columns = {"code", "name", "synonyms"}
missing_columns = required_columns - set(df_concepts.columns)
if missing_columns:
    raise ValueError(f"Missing required concept columns: {sorted(missing_columns)}")

concepts, seen_codes = [], set()
for _, row in df_concepts.iterrows():
    code = normalize_code(row["code"])
    if not code or code in seen_codes:
        continue
    seen_codes.add(code)
    name = normalize_alias(row["name"])
    if not name:
        continue
    concepts.append({
        "id": code,
        "text": name,
        "aliases": make_aliases(code, name, row["synonyms"]),
        "group": concept_group(code),
    })

# Remove aliases shared by multiple concepts, except each concept's canonical name.
alias_to_codes = defaultdict(set)
for concept in concepts:
    for alias in concept["aliases"]:
        alias_to_codes[alias.casefold()].add(concept["id"])
for concept in concepts:
    canonical = concept["text"].casefold()
    concept["aliases"] = [
        alias for alias in concept["aliases"]
        if alias.casefold() == canonical or len(alias_to_codes[alias.casefold()]) == 1
    ]

pd.DataFrame([
    {
        "code": concept["id"],
        "name": concept["text"],
        "synonyms": repr(concept["aliases"][1:]),
        "aliases_used": "|".join(concept["aliases"]),
    }
    for concept in concepts
]).to_csv(ALIAS_TABLE_OUT, index=False)

with open(TRAIN_JSON) as f:
    train_samples = json.load(f)
with open(DEV_JSON) as f:
    dev_samples = json.load(f)
with open(VAL_JSON) as f:
    val_samples = json.load(f)
for samples in (train_samples, dev_samples, val_samples):
    for sample in samples:
        sample["label"] = int(sample["label"] >= 3)

print(f"Concepts: {len(concepts):,}; aliases: {sum(len(c['aliases']) for c in concepts):,}")
print("Alias-count summary:", pd.Series([len(c["aliases"]) for c in concepts]).describe().round(2).to_dict())
print("Outcome labels:", Counter(sample["label"] for sample in train_samples))


In [ ]:
class TextDataset(Dataset):
    def __init__(self, samples): self.samples = samples
    def __len__(self): return len(self.samples)
    def __getitem__(self, i): return self.samples[i]


def make_loader(samples, tokenizer, batch_size=4, max_length=512, shuffle=False, include_concepts=False):
    concept_to_idx = {c["id"]: i for i, c in enumerate(concepts)}

    def annotation_code(annotation):
        if isinstance(annotation, dict): raw = annotation.get("code", annotation.get("id"))
        elif isinstance(annotation, (list, tuple)) and len(annotation) >= 2: raw = annotation[1]
        else: raw = None
        return normalize_code(raw) if raw is not None else ""

    def collate(batch):
        encoded = tokenizer(
            [s["txt"] for s in batch], padding=True, truncation=True,
            max_length=max_length, return_tensors="pt",
        )
        encoded["labels"] = torch.tensor([s["label"] for s in batch], dtype=torch.long)
        if include_concepts:  # evaluation only
            y = torch.zeros(len(batch), len(concepts), dtype=torch.bool)
            for i, sample in enumerate(batch):
                for annotation in sample.get("concepts", []):
                    j = concept_to_idx.get(annotation_code(annotation))
                    if j is not None: y[i, j] = True
            encoded["concept_labels"] = y
        return encoded

    return DataLoader(
        TextDataset(samples), batch_size=batch_size, shuffle=shuffle,
        num_workers=0, pin_memory=True, collate_fn=collate,
    )

In [ ]:
STOP_TOKENS = {
    "of", "and", "or", "the", "a", "an", "in", "on", "with", "without",
    "other", "specified", "unspecified", "elsewhere", "classified", "not",
}


def informative_token(token: str) -> bool:
    token = token.replace("##", "").strip().lower()
    token = re.sub(r"[^a-z0-9]+", "", token)
    return bool(token) and token not in STOP_TOKENS


@torch.no_grad()
def build_cls_embeddings(texts, tokenizer, encoder, device, batch_size=32, max_length=64):
    rows = []
    encoder.eval()
    for start in range(0, len(texts), batch_size):
        tok = tokenizer(
            texts[start:start + batch_size], padding=True, truncation=True,
            max_length=max_length, return_tensors="pt",
        ).to(device)
        rows.append(encoder(**tok).last_hidden_state[:, 0].cpu())
    return torch.cat(rows)


@torch.no_grad()
def build_prototype_bank(
    concepts, tokenizer, encoder, device,
    max_aliases=3, max_tokens=16, batch_size=32,
):
    """Token embeddings for canonical descriptions and aliases: (C,A,T,H)."""
    flat, locations = [], []
    for concept_idx, concept in enumerate(concepts):
        aliases = concept["aliases"][:max_aliases] or [concept["text"]]
        for alias_idx, alias in enumerate(aliases):
            flat.append(alias); locations.append((concept_idx, alias_idx))

    hidden_size = encoder.config.hidden_size
    dtype = torch.float16 if device.type == "cuda" else torch.float32
    bank = torch.zeros(len(concepts), max_aliases, max_tokens, hidden_size, dtype=dtype)
    weights = torch.zeros(len(concepts), max_aliases, max_tokens, dtype=torch.float32)
    encoder.eval()

    for start in range(0, len(flat), batch_size):
        texts = flat[start:start + batch_size]
        tok = tokenizer(
            texts, padding=True, truncation=True, max_length=max_tokens + 2,
            return_tensors="pt",
        ).to(device)
        hidden = encoder(**tok).last_hidden_state
        for row_idx, (concept_idx, alias_idx) in enumerate(locations[start:start + batch_size]):
            ids = tok["input_ids"][row_idx]
            valid = tok["attention_mask"][row_idx].bool()
            for special_id in tokenizer.all_special_ids:
                valid &= ids != special_id
            positions = torch.where(valid)[0]
            tokens = tokenizer.convert_ids_to_tokens(ids[positions].tolist())
            keep = [i for i, token in enumerate(tokens) if informative_token(token)]
            if not keep: keep = list(range(len(tokens)))
            positions = positions[keep][:max_tokens]
            n = len(positions)
            if n:
                bank[concept_idx, alias_idx, :n] = hidden[row_idx, positions].detach().cpu().to(dtype)
                weights[concept_idx, alias_idx, :n] = 1.0

    return bank, weights


@dataclass
class AVOOutput:
    logits: torch.Tensor
    token_logits: torch.Tensor
    A: torch.Tensor
    V: torch.Tensor
    O: torch.Tensor
    sim: torch.Tensor
    A_pool: torch.Tensor
    AV_pool: torch.Tensor
    concept_logits: torch.Tensor


class LateInteractionAVOHead(nn.Module):
    """Whole-phrase coverage using token-level late interaction over fixed aliases."""
    def __init__(
        self, concept_emb, prototype_bank, prototype_weights,
        dv=256, num_outputs=2, match_margin=0.65,
        match_temperature=0.05, concept_chunk_size=32,
    ):
        super().__init__()
        C, H = concept_emb.shape
        self.C, self.H, self.dv = C, H, dv
        self.match_margin = float(match_margin)
        self.match_temperature = float(match_temperature)
        self.concept_chunk_size = int(concept_chunk_size)
        self.register_buffer("concept_emb", concept_emb.detach().clone())
        self.register_buffer("prototype_bank", prototype_bank.detach().clone())
        self.register_buffer("prototype_weights", prototype_weights.detach().clone())

        # Shared, frozen, identity projection preserves exact matches and SapBERT geometry.
        self.Wmatch = nn.Linear(H, H, bias=False)
        nn.init.eye_(self.Wmatch.weight)
        for p in self.Wmatch.parameters(): p.requires_grad = False

        self.Wv = nn.Linear(H, dv, bias=False)
        self.O = nn.Parameter(torch.randn(dv, num_outputs) * 0.02)
        self.bias = nn.Parameter(torch.zeros(num_outputs))

    def forward(self, token_embs, token_mask):
        q = F.normalize(self.Wmatch(token_embs), dim=-1)
        B, L, _ = q.shape
        A_real = q.new_empty(B, L, self.C)
        raw_presence = q.new_empty(B, self.C)
        note_mask = token_mask[:, :, None, None, None]

        for start in range(0, self.C, self.concept_chunk_size):
            end = min(start + self.concept_chunk_size, self.C)
            proto = self.prototype_bank[start:end].to(device=q.device, dtype=q.dtype)
            weight = self.prototype_weights[start:end].to(q.device)
            k = F.normalize(self.Wmatch(proto), dim=-1)

            # sim: (batch, note_token, concept, alias, alias_token)
            sim = torch.einsum("blh,cath->blcat", q, k)
            sim = sim.masked_fill(~note_mask, -1e4)

            valid_proto = weight > 0
            token_support = sim.masked_fill(
                ~valid_proto[None, None], -1e4
            ).amax(dim=(-1, -2))
            A_real[:, :, start:end] = torch.sigmoid(
                (token_support - self.match_margin) / self.match_temperature
            ).masked_fill(~token_mask.unsqueeze(-1), 0.0)

            # Each informative alias token must be covered somewhere in the note.
            best_note = sim.max(dim=1).values                    # (B,c,A,T)
            denom = weight.sum(dim=-1).clamp(min=1.0)
            alias_score = (best_note * weight[None]).sum(dim=-1) / denom[None]
            alias_score = alias_score.masked_fill(
                ~(valid_proto.any(dim=-1))[None], -1e4
            )
            raw_presence[:, start:end] = alias_score.max(dim=-1).values

        concept_logits = (raw_presence - self.match_margin) / self.match_temperature
        presence = torch.sigmoid(concept_logits)

        # Retain a zero-contribution NULL row for compatibility with the original AVO layer.
        A_null = 1.0 - A_real.max(dim=-1, keepdim=True).values
        A = torch.cat([A_null, A_real], dim=-1)
        pool_null = 1.0 - presence.max(dim=-1, keepdim=True).values
        A_pool = torch.cat([pool_null, presence], dim=-1)

        V_real = self.Wv(self.concept_emb)
        V = torch.cat([V_real.new_zeros(1, self.dv), V_real], dim=0)
        AV_pool = A_pool @ V
        logits = AV_pool @ self.O + self.bias

        token_logits = torch.logit(A.clamp(1e-6, 1 - 1e-6))
        return AVOOutput(
            logits=logits, token_logits=token_logits, A=A, V=V, O=self.O,
            sim=A, A_pool=A_pool, AV_pool=AV_pool,
            concept_logits=concept_logits,
        )


class LateInteractionAVOModel(nn.Module):
    def __init__(self, encoder, head, tokenizer):
        super().__init__()
        self.text_encoder, self.head = encoder, head
        self.special_ids = tokenizer.all_special_ids
        for p in self.text_encoder.parameters(): p.requires_grad = False

    def train(self, mode=True):
        super().train(mode)
        self.text_encoder.eval()  # keep note/prototype embedding spaces identical
        return self

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        kwargs = {"input_ids": input_ids, "attention_mask": attention_mask}
        if token_type_ids is not None: kwargs["token_type_ids"] = token_type_ids
        token_embs = self.text_encoder(**kwargs).last_hidden_state
        token_mask = attention_mask.bool()
        for token_id in self.special_ids: token_mask &= input_ids != token_id
        return self.head(token_embs, token_mask), token_mask

In [ ]:
def group_lasso(beta, groups, lambda_=1e-3):
    penalty = beta.new_zeros(())
    for group in groups:
        idx = torch.as_tensor(group, device=beta.device, dtype=torch.long) + 1
        penalty = penalty + math.sqrt(len(group)) * beta.index_select(0, idx).norm()
    return lambda_ * penalty


def build_groups(concepts):
    by_group = defaultdict(list)
    for i, concept in enumerate(concepts): by_group[concept["group"]].append(i)
    return list(by_group.values())


@torch.no_grad()
def evaluate_outcome(model, loader, device):
    model.eval(); ys, ps = [], []
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items() if isinstance(v, torch.Tensor)}
        out, _ = model(
            batch["input_ids"], batch["attention_mask"], batch.get("token_type_ids")
        )
        ys.append(batch["labels"].cpu().numpy())
        ps.append(torch.softmax(out.logits, -1)[:, 1].cpu().numpy())
    y, p = np.concatenate(ys), np.concatenate(ps)
    return {"AUROC": roc_auc_score(y, p), "AUPR": average_precision_score(y, p)}


@torch.no_grad()
def evaluate_grounding(model, loader, device, ks=(1, 5, 10), contribution=False):
    model.eval(); hits = {k: 0 for k in ks}; n = 0
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items() if isinstance(v, torch.Tensor)}
        out, _ = model(
            batch["input_ids"], batch["attention_mask"], batch.get("token_type_ids")
        )
        scores = out.A_pool[:, 1:]
        if contribution:
            beta = (out.V @ out.O)[1:]
            scores = scores * (beta[:, 1] - beta[:, 0]).unsqueeze(0)
        labels = batch["concept_labels"].bool()
        valid = labels.any(dim=1)
        if not valid.any(): continue
        top = scores[valid].topk(max(ks), dim=1).indices
        labels = labels[valid]; n += int(valid.sum())
        for k in ks:
            hits[k] += int(labels.gather(1, top[:, :k]).any(dim=1).sum())
    prefix = "contributor" if contribution else "presence"
    return {"n_labeled_notes": n, **{f"{prefix}_hit@{k}": hits[k] / max(n, 1) for k in ks}}


@torch.no_grad()
def exact_match_test(model, tokenizer, concepts, device, max_examples=512, batch_size=8):
    """Ontology-only unit test: feed exact stored aliases as notes and recover their code."""
    examples = [(alias, j) for j, c in enumerate(concepts) for alias in c["aliases"][:2]]
    rng = random.Random(SEED); rng.shuffle(examples)
    if max_examples is not None: examples = examples[:max_examples]
    ranks, probabilities = [], []
    model.eval()
    for start in range(0, len(examples), batch_size):
        chunk = examples[start:start + batch_size]
        tok = tokenizer(
            [x[0] for x in chunk], padding=True, truncation=True,
            max_length=64, return_tensors="pt",
        ).to(device)
        out, _ = model(**tok)
        scores = out.A_pool[:, 1:]
        target = torch.tensor([x[1] for x in chunk], device=device)
        target_prob = scores.gather(1, target[:, None]).squeeze(1)
        rank = 1 + (scores > target_prob[:, None]).sum(dim=1)
        ranks.extend(rank.cpu().tolist()); probabilities.extend(target_prob.cpu().tolist())
    ranks = np.asarray(ranks); probabilities = np.asarray(probabilities)
    return {
        "n": len(ranks), "hit@1": float((ranks <= 1).mean()),
        "hit@5": float((ranks <= 5).mean()), "hit@10": float((ranks <= 10).mean()),
        "median_target_probability": float(np.median(probabilities)),
    }

In [ ]:
MODEL_NAME = "cambridgeltl/SapBERT-from-PubMedBERT-fulltext"
BATCH_SIZE = 4
MAX_LENGTH = 512
DV = 256
OUTCOME_WARMUP_EPOCHS = 1
OUTCOME_EPOCHS = 2
OUTCOME_LR = 1e-5
MATCH_MARGIN = 0.65
MATCH_TEMPERATURE = 0.05
CONCEPT_CHUNK_SIZE = 32
MAX_ALIASES = 3
MAX_ALIAS_TOKENS = 16
EXACT_MATCH_EXAMPLES = 512  # set None to test every canonical description

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
encoder = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)
train_loader = make_loader(train_samples, tokenizer, BATCH_SIZE, MAX_LENGTH, shuffle=True)
dev_loader = make_loader(dev_samples, tokenizer, BATCH_SIZE, MAX_LENGTH)
val_loader = make_loader(val_samples, tokenizer, BATCH_SIZE, MAX_LENGTH)
dev_grounding_loader = make_loader(dev_samples, tokenizer, BATCH_SIZE, MAX_LENGTH, include_concepts=True)
val_grounding_loader = make_loader(val_samples, tokenizer, BATCH_SIZE, MAX_LENGTH, include_concepts=True)

class BlackBoxLM(nn.Module):
    def __init__(self, encoder):
        super().__init__(); self.encoder = encoder
        self.head = nn.Linear(encoder.config.hidden_size, 2)
    def forward(self, input_ids, attention_mask, token_type_ids=None):
        kwargs = {"input_ids": input_ids, "attention_mask": attention_mask}
        if token_type_ids is not None: kwargs["token_type_ids"] = token_type_ids
        hidden = self.encoder(**kwargs).last_hidden_state
        return self.head(hidden[:, 0])

blackbox = BlackBoxLM(encoder).to(DEVICE)
optimizer = torch.optim.AdamW(blackbox.parameters(), lr=1e-5)
for epoch in range(OUTCOME_WARMUP_EPOCHS):
    blackbox.train(); total = 0.0
    for batch in train_loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items() if isinstance(v, torch.Tensor)}
        logits = blackbox(batch["input_ids"], batch["attention_mask"], batch.get("token_type_ids"))
        loss = F.cross_entropy(logits, batch["labels"])
        optimizer.zero_grad(set_to_none=True); loss.backward(); optimizer.step()
        total += loss.item()
    print(f"Outcome warm-up {epoch + 1}: loss={total / len(train_loader):.4f}")

In [ ]:
# Build the fixed ontology matcher after outcome warm-up.
encoder = blackbox.encoder
concept_emb = build_cls_embeddings(
    [c["text"] for c in concepts], tokenizer, encoder, DEVICE,
    batch_size=32, max_length=64,
).to(DEVICE)
prototype_bank, prototype_weights = build_prototype_bank(
    concepts, tokenizer, encoder, DEVICE,
    max_aliases=MAX_ALIASES, max_tokens=MAX_ALIAS_TOKENS, batch_size=32,
)

head = LateInteractionAVOHead(
    concept_emb=concept_emb,
    prototype_bank=prototype_bank,
    prototype_weights=prototype_weights,
    dv=DV,
    num_outputs=2,
    match_margin=MATCH_MARGIN,
    match_temperature=MATCH_TEMPERATURE,
    concept_chunk_size=CONCEPT_CHUNK_SIZE,
).to(DEVICE)
model = LateInteractionAVOModel(encoder, head, tokenizer).to(DEVICE)
del concept_emb, prototype_bank, prototype_weights
if torch.cuda.is_available(): torch.cuda.empty_cache()

print("Exact-match diagnostic:", exact_match_test(
    model, tokenizer, concepts, DEVICE, max_examples=EXACT_MATCH_EXAMPLES
))
print("Dev presence grounding before outcome training:",
      evaluate_grounding(model, dev_grounding_loader, DEVICE))

In [ ]:
groups = build_groups(concepts)
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad], lr=OUTCOME_LR
)

for epoch in range(1, OUTCOME_EPOCHS + 1):
    model.train(); total = 0.0
    for batch in train_loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items() if isinstance(v, torch.Tensor)}
        out, _ = model(
            batch["input_ids"], batch["attention_mask"], batch.get("token_type_ids")
        )
        beta = out.V @ out.O
        loss = F.cross_entropy(out.logits, batch["labels"]) + group_lasso(beta, groups, 1e-3)
        optimizer.zero_grad(set_to_none=True); loss.backward(); optimizer.step()
        total += loss.item()

    print(f"Epoch {epoch}: loss={total / len(train_loader):.4f}")
    print("  outcome:", evaluate_outcome(model, dev_loader, DEVICE))
    print("  presence:", evaluate_grounding(model, dev_grounding_loader, DEVICE))
    print("  contributor:", evaluate_grounding(
        model, dev_grounding_loader, DEVICE, contribution=True
    ))

print("Final validation/test outcome:", evaluate_outcome(model, val_loader, DEVICE))
print("Final validation/test presence grounding:",
      evaluate_grounding(model, val_grounding_loader, DEVICE))
print("Final validation/test contributor grounding:",
      evaluate_grounding(model, val_grounding_loader, DEVICE, contribution=True))